# 1장 1강: 텍스트 데이터의 이해와 전처리 — 실습문제


## 실습 목표

- 표 형태로 저장된 데이터와 그 안에 들어 있는 자연어 본문의 특징을 구분할 수 있다.
- 리뷰 본문, 메타데이터, 기존 라벨의 역할을 설명할 수 있다.
- 앞뒤 공백 제거와 공백 기준 토큰화를 수행하고 두 작업의 차이를 설명할 수 있다.
- 형태소 분석 예시를 읽고, 명사만 남겼을 때 사라지는 정보를 설명할 수 있다.
- 키워드 분석과 LLM 기반 분석에 적합한 입력을 구분하고 리뷰와 관련 정보를 구조화할 수 있다.

## 실습 환경 / 데이터

- Python 3, Jupyter Notebook 또는 VS Code의 노트북 환경, pandas
- 제공 파일: `reviews.csv`, `review_labels.csv`, `aspect_labels.csv`
- 노트북과 CSV 3개를 같은 폴더에 놓습니다. 다른 폴더라면 준비 코드의 `DATA_DIR`만 수정합니다.
- 위에서 아래로 순서대로 실행합니다. 필수 2와 과제 1은 필수 1에서 만든 `documents`를 사용합니다.

| 파일 | 한 행의 의미 | 주요 컬럼과 역할 |
|---|---|---|
| `reviews.csv` | 리뷰 한 개 | `review_id`: 식별자, `review_text`: 본문, `source`: 출처, `domain`: 분야, `category`: 상품 분류, `product_name`: 상품명 |
| `review_labels.csv` | 리뷰 한 개에 대한 기존 전체 감정 코드 | `review_id`, `sentiment_code` |
| `aspect_labels.csv` | 리뷰 안의 속성별 주석 한 개 | `review_id`, `aspect_no`: 리뷰 안의 주석 번호, `aspect`: 속성, `evidence_text`: 근거 문장, `sentiment_code`: 해당 주석의 감정 코드 |

`review_id`가 같으면 같은 리뷰에 관한 정보입니다. 속성별 파일은 같은 리뷰에 여러 행이 있을 수 있으며, 같은 속성이 반복될 수도 있습니다. **속성별 행 수를 리뷰 개수로 읽지 않습니다.**

이 실습의 문서 단위는 **리뷰 하나**입니다. `sentiment_code`는 이미 제공된 범주 코드이며, 이번 실습에서 모델로 예측한 값이나 평점이 아닙니다. 코드의 평균을 만족도로 해석하지 않습니다. 별도의 라벨 정의서가 없으므로 숫자 전체에 감정 이름을 일괄 부여하지 않고 실제 근거 표현을 읽어 판단합니다. 특히 코드 `0`과 결측치는 서로 다릅니다.

### 이번 강의의 범위

강의의 정제·토큰화·형태소 분석 개념, 목적에 따른 전처리 선택, 문서 구조화를 연습합니다. TF-IDF·임베딩·LLM은 **역할과 입력·출력 형태를 구분**합니다. 해당 모델을 계산·학습하거나 API를 호출하는 과제는 없습니다.

형태소 분석은 교안의 짧은 예시를 해석하는 방식입니다. 실제 분석기를 설치하거나 실행하지 않습니다. 아래에서 제공하는 수동 분해 예시는 형태소 분석기 출력으로 오해하지 않도록 구분합니다.


## 시작 전에 읽는 개념 정리

이번 강의의 목표는 **리뷰의 뜻을 보존하면서, 분석하기 편한 형태로 준비하는 것**입니다. 예를 들어 “옷은 예쁘지만 얇아서 추워요”에서 ‘옷’만 남기면 무엇에 관한 글인지는 알 수 있지만, 왜 아쉬웠는지는 알기 어렵습니다. 어떤 정보를 남길지는 분석 목적을 먼저 생각하고 정합니다.

| 용어 | 쉽게 풀어쓴 뜻 | 이번 실습에서의 예 |
|---|---|---|
| 텍스트 데이터 | 사람이 말이나 글로 표현한 자료 | 상품을 사용한 경험을 적은 리뷰 본문 |
| 문서 | 분석에서 하나로 다룰 텍스트 묶음 | 이번에는 리뷰 한 개. 문장 하나마다 문서로 나누지 않음 |
| DataFrame | 행과 열로 이루어진 표 | `reviews`는 리뷰와 관련 정보를 담은 표 |
| 컬럼·행 | 컬럼은 정보의 종류, 행은 기록 하나 | `review_text`는 본문 컬럼, reviews의 한 행은 리뷰 하나 |
| 문자열 | 글자를 순서대로 담은 값 | `"상품이 좋아요."` |
| 메타데이터 | 본문이 어떤 자료인지 설명하는 부가 정보 | 출처·분야·상품 분류·상품명 |
| 식별자(ID) | 같은 대상을 찾기 위한 이름표 | `review_id`. 숫자가 커도 더 좋은 리뷰라는 뜻은 아님 |
| 라벨 | 자료에 미리 붙여 둔 분류·판단 정보 | 전체 리뷰 또는 특정 속성의 `sentiment_code` |
| 속성·근거 문장 | 평가한 대상과 그 판단의 근거 표현 | 속성은 ‘가격’, 근거는 ‘저렴하게 구매해서 만족’ |
| 결측치 | 값이 기록되지 않은 상태 | 전체 라벨이 없는 경우. 숫자 0과 다름 |
| 정제 | 목적에 맞게 불필요한 부분을 정리하는 작업 | 이번에는 본문 앞뒤 공백만 제거 |
| 토큰·토큰화 | 나눈 조각과 조각으로 나누는 작업 | 이번에는 공백을 기준으로 나눈 문자열 목록 |
| 형태소·품사 | 의미·문법 기능을 가진 단위와 그 역할 구분 | `상품이`를 명사 `상품`과 조사 `이`로 설명 |
| 문맥 | 앞뒤 표현·대상·조건이 함께 만드는 의미 | ‘얇다’가 여름용 장점인지 추운 날 단점인지 원문으로 확인 |
| 파생 컬럼 | 기존 값으로부터 새로 만든 컬럼 | 본문에서 만든 `clean_text`, `char_count` |

**표의 구조와 본문의 구조를 구분하세요.** 모든 행에 `review_text`라는 칸이 있어도 그 안의 문장 수·단어·표현은 제각각입니다. 표에 저장했다는 사실만으로 자연어의 뜻이 숫자처럼 자동으로 정리되지는 않습니다.

### 세 가지 전처리를 구분하기

| 구분 | 이번에 하는 일 | 결과 형태 | 남는 정보와 한계 |
|---|---|---|---|
| 공백 정제 | 문자열 양 끝의 불필요한 공백 제거 | 문자열 | 문장 내부 표현은 유지 |
| 공백 토큰화 | 공백 문자를 기준으로 나누기 | 문자열을 담은 목록 | 조사·어미·문장부호가 토큰에 붙어 있을 수 있음 |
| 형태소 분석 개념 확인 | 제공된 `상품/이`, `상품/은` 예시 읽기 | 사람이 해석한 분해 예시 | 실제 형태소 분석기 실행 결과가 아님 |

공백 토큰화는 명사만 추출하는 작업이 아닙니다. 또한 실제 LLM이 내부에서 사용하는 토큰 수를 이번 `token_count`로 알 수 있는 것도 아닙니다. **토큰 수를 말할 때는 어떤 기준으로 나눴는지 함께 말해야 합니다.**

### 이번 강의의 작업 순서

파일 읽기 → 구조와 결측치 확인 → 리뷰 100개와 연결된 주석 선택 → 본문 정제·공백 토큰화 → 원문·근거 비교 → 다른 리뷰를 같은 방식으로 정리합니다. TF-IDF·임베딩·LLM은 목적과 출력 형태를 구분하는 데까지만 다룹니다.

### 코드를 읽기 위한 기초 문법

| 코드 표현 | 읽는 방법 | 확인할 점 |
|---|---|---|
| `이름 = 값` | 오른쪽 값을 왼쪽 이름에 저장 | `=`는 저장, `==`는 같은지 비교 |
| `table["review_text"]` | 표에서 본문 컬럼 하나 선택 | 선택 결과는 한 컬럼의 값들 |
| `table[["review_id", "review_text"]]` | 두 컬럼을 가진 표 선택 | 안쪽 대괄호는 컬럼 이름 목록 |
| `table[조건]` | 조건이 True인 행만 선택 | 원본 행 위치와 조건의 대응을 유지 |
| `.copy()` | 별도 작업용 복사본 만들기 | 원본과 작업 결과를 구분하기 위해 사용 |
| `.iloc[0]` / `.iloc[1]` | 첫 번째 / 두 번째 위치의 행 선택 | 위치 번호는 0부터 시작하며 리뷰 ID가 아님 |
| `len(table)` / `len(목록)` | 표의 행 수 / 목록의 원소 수 | 무엇의 개수를 세는지 먼저 확인 |
| `print(값)` | 화면에 결과 표시 | 화면에 보이는 것이 별도 파일 저장을 뜻하지는 않음 |
| `for name, table in ...:` | 준비한 이름·표의 쌍을 하나씩 꺼내 반복 | 들여쓴 줄들이 각 표에 대해 실행됨 |
| `def ...` / `return ...` | 함수 정의 / 호출한 곳에 결과 돌려주기 | 함수 정의 후 실제 사용하려면 호출 필요 |

처음부터 모든 문법을 외우기보다 **지금 선택한 대상이 표인지, 한 행인지, 문자열인지, 목록인지**를 확인하세요. 같은 `len()`이라도 대상에 따라 세는 단위가 달라집니다. 준비 코드와 조회 함수는 제공된 내용을 실행하고 역할을 이해하면 됩니다.

## 실습 준비

아래 코드는 데이터 확인을 위한 **제공 코드**입니다. 수정 없이 실행한 뒤 출력 내용을 읽으세요.

1. 파일별 행·열 수, 컬럼명, 자료형, 결측치를 확인합니다.
2. 리뷰 본문과 상품명 등 부가 정보의 역할을 구분합니다.
3. 원본 순서의 앞 100개 리뷰를 `sample_reviews`로 선택합니다.
4. 선택된 리뷰 ID에 해당하는 전체 라벨과 속성별 주석을 준비합니다.
5. 본문 길이, 출처·분야 분포와 기존 감정 코드 분포를 확인합니다.

> 앞 100개는 빠르게 연습하기 위한 고정 샘플입니다. 무작위 표본이 아니므로 전체 고객이나 전체 도메인의 특성을 대표한다고 해석하지 않습니다. 라벨이 비어 있더라도 본문이 있으면 전처리 연습에 사용할 수 있습니다.


In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(".")  # 노트북과 CSV 3개가 같은 폴더에 있는 경우
# 예: DATA_DIR = Path("C:/Users/jihun/OneDrive/Desktop/텍스트분석/DATA")

reviews = pd.read_csv(DATA_DIR / "reviews.csv")
review_labels = pd.read_csv(DATA_DIR / "review_labels.csv")
aspect_labels = pd.read_csv(DATA_DIR / "aspect_labels.csv")

for name, table in [("reviews", reviews),
                    ("review_labels", review_labels),
                    ("aspect_labels", aspect_labels)]:
    print(f"\n[{name}] {table.shape[0]:,}행, {table.shape[1]}열")
    print("컬럼:", table.columns.tolist())
    print(pd.DataFrame({"자료형": table.dtypes.astype(str),
                        "결측치 수": table.isna().sum()}).to_string())

print("\n리뷰 ID의 유일성:", reviews["review_id"].is_unique)
print("전체 라벨 ID의 유일성:", review_labels["review_id"].is_unique)
print("전체 라벨의 리뷰 ID 집합 일치:",
      set(reviews["review_id"]) == set(review_labels["review_id"]))
print("속성 주석의 리뷰 ID 집합 일치:",
      set(reviews["review_id"]) == set(aspect_labels["review_id"]))

sample_reviews = reviews.head(100).copy()
sample_ids = sample_reviews["review_id"]
sample_review_labels = review_labels[
    review_labels["review_id"].isin(sample_ids)
].copy()
sample_aspect_labels = aspect_labels[
    aspect_labels["review_id"].isin(sample_ids)
].copy()

print("\n샘플별 행 수:", len(sample_reviews),
      len(sample_review_labels), len(sample_aspect_labels))
print("\n리뷰 메타데이터 앞 3행:")
print(sample_reviews.drop(columns="review_text").head(3).to_string(index=False))
print("\n첫 리뷰 본문:")
print(sample_reviews.iloc[0]["review_text"])
print("\n원문 글자 수의 기초 통계량:")
print(sample_reviews["review_text"].str.len().describe().round(2).to_string())
print("\n샘플 출처 분포:")
print(sample_reviews["source"].value_counts(dropna=False).to_string())
print("\n샘플 분야 분포:")
print(sample_reviews["domain"].value_counts(dropna=False).to_string())
print("\n전체 리뷰 라벨 코드 분포(결측 포함):")
print(review_labels["sentiment_code"].value_counts(dropna=False).to_string())


[reviews] 99,992행, 6열
컬럼: ['review_id', 'source', 'domain', 'category', 'product_name', 'review_text']
                자료형  결측치 수
review_id     int64      0
source          str      0
domain          str      0
category        str      0
product_name    str     16
review_text     str      0

[review_labels] 99,992행, 2열
컬럼: ['review_id', 'sentiment_code']
                    자료형  결측치 수
review_id         int64      0
sentiment_code  float64   1143

[aspect_labels] 373,167행, 5열
컬럼: ['review_id', 'aspect_no', 'aspect', 'evidence_text', 'sentiment_code']
                  자료형  결측치 수
review_id       int64      0
aspect_no       int64      0
aspect            str      0
evidence_text     str      0
sentiment_code  int64      0

리뷰 ID의 유일성: True
전체 라벨 ID의 유일성: True
전체 라벨의 리뷰 ID 집합 일치: True
속성 주석의 리뷰 ID 집합 일치: True

샘플별 행 수: 100 100 715

리뷰 메타데이터 앞 3행:
 review_id source domain category  product_name
   1024338    SNS     패션     여성의류   OO 경량 다운 자켓
   1024477    SNS     패션     여성의류 OO 아** 구스코트  

### 준비 코드와 출력의 연결

`DATA_DIR = Path(".")`의 점은 현재 작업 폴더를 뜻합니다. 실행 환경에 따라 노트북이 저장된 폴더와 다를 수도 있으므로, 다른 위치에 파일이 있으면 CSV 세 개가 있는 폴더를 DATA_DIR에 지정합니다. `DATA_DIR / "reviews.csv"`의 `/`는 경로를 이어 붙이는 기호입니다.

준비 코드는 세 파일을 읽은 뒤 행·열·결측치를 표시하고, 앞 100개 리뷰의 ID를 기준으로 관련 라벨을 고릅니다. 각 파일에서 단순히 앞 100행을 가져오는 방식이 아닙니다. 속성 파일에는 같은 리뷰의 여러 주석이 들어 있기 때문입니다.

출력을 읽을 때는 ① 각 표에서 한 행이 뜻하는 것, ② 빈 값이 어느 컬럼에 있는지, ③ 선택된 리뷰와 주석의 대응, ④ 본문의 길이와 출처를 차례로 확인합니다. True는 조건이 맞다는 뜻이며 긍정 감정이라는 뜻이 아닙니다. 코드별 자세한 설명과 실제 수치 해석은 정답 및 해설의 준비 코드 설명에 있습니다.

---
## 필수 1. 리뷰 본문 정제와 공백 기준 토큰화

### 문제 1-1. 본문을 보존하면서 분석용 텍스트를 준비하기

#### 문제 설명

리뷰의 주요 단어를 살펴보기 전에 본문을 정리하려고 합니다. 원문을 보존하고, 분석용 컬럼을 따로 만들어 전처리 전후를 비교하세요.

#### 요구사항

1. `sample_reviews`에서 `review_text`가 결측인 행만 제외하고 복사하여 `documents`를 만드세요. 다른 컬럼의 결측 때문에 리뷰를 제거하지 마세요.
2. `review_text`의 **앞뒤 공백만** 제거한 `clean_text`를 만드세요. 문장 내부의 공백·문장부호·숫자·영문·부정 표현은 그대로 유지하세요.
3. 정제 결과가 빈 문자열인 행을 제외하고 인덱스를 정리하세요.
4. `clean_text`를 인자 없는 `split()`에 해당하는 방법으로 나누어 `space_tokens`에 저장하세요. 각 행의 값은 토큰 목록이어야 합니다.
5. `clean_text`의 글자 수를 `char_count`, 공백 기준 토큰 수를 `token_count`에 저장하세요.
6. 준비 샘플 수, 본문 결측 수, 공백뿐인 본문 수, 최종 문서 수, 앞뒤 공백 제거로 본문이 달라진 문서 수를 출력하세요. 글자 수·토큰 수의 최솟값, 평균, 최댓값도 출력하세요.
7. 첫 문서의 원문과 정제문 앞뒤 공백이 보이도록 `repr()`로 출력하고, 앞 10개 토큰을 출력하세요.

#### 해석 질문

**Q1.** 리뷰를 DataFrame에 저장하면 리뷰 본문 자체도 숫자 데이터처럼 일정한 구조를 갖게 되나요?

**Q2.** `strip()`과 `split()`은 각각 정제와 토큰화 중 어떤 작업인가요? 공백 기준 토큰 수를 형태소 수라고 불러도 되나요?

**Q3.** 문장부호, 숫자, 부정 표현을 일괄 삭제하지 않은 이유는 무엇인가요?

**Q4.** 본문이 있는 리뷰의 상품명이나 전체 라벨이 비어 있다는 이유만으로 그 리뷰를 반드시 제거해야 하나요?

#### 제출 결과

- 요구사항 1~7의 코드와 출력
- 원문 및 파생 컬럼이 함께 있는 `documents`
- Q1~Q4 답변


### 필수 1 풀이를 시작하기 전에

먼저 본문이 있는지 확인하고, 그다음 공백을 정리한 결과가 비었는지 확인합니다. 순서를 바꾸면 공백만 있는 본문을 놓칠 수 있습니다. 새 컬럼은 원문 옆에 추가하여 비교하세요.

- `isna()`는 결측 검사, `eq("")`는 빈 문자열 검사입니다.
- `str.strip()`은 문자열 정제, `str.split()`은 토큰 목록 생성입니다.
- `str.len()`으로 무엇을 세는지는 적용 대상이 문자열인지 목록인지에 따라 달라집니다.
- 출력할 때 문서 수·글자 수·토큰 수의 단위를 함께 적으세요.

답변에는 수치뿐 아니라 그 수치로 확인할 수 있는 사실을 적습니다. 문제의 최종 결과와 전체 풀이 코드는 정답 및 해설에서 확인할 수 있습니다.

In [33]:
# 필수 1 코드를 작성하세요.

# 결측만 제외
documents = sample_reviews.dropna(subset=['review_text']).copy()

# 앞뒤 공백 제거
documents['clean_text'] = documents['review_text'].str.strip()



# 정제한 뒤 빈 문자열이 된 행을 셈
blank_body_count = documents['clean_text'].eq('').sum()
# 빈 문자열 제외하고 인덱스 정리
documents = documents[documents['clean_text']!=''].reset_index(drop=True)

# 토큰, 글자 수, 토큰 수
documents['space_tokens'] = documents['clean_text'].str.split()
documents['char_count'] = documents['clean_text'].str.len()
documents['token_count']=documents['space_tokens'].str.len()

print('준비 샘플 수:',len(sample_reviews),"개")
print('본문 결측 수:',sample_reviews['review_text'].isna().sum(),'개')
print('공백뿐인 본문 수:',blank_body_count,'개')
print('최종 문서 수:',len(documents),'개')
print('공백 제거로 달라진 문서 수;',(documents['review_text']!=documents['clean_text']).sum(),'개')
print(documents[['char_count','token_count']].agg(['min','mean','max']))

print(repr(documents.loc[0,'review_text']))
print(repr(documents.loc[0,'clean_text']))
print(documents.loc[0,'space_tokens'][:10]);

준비 샘플 수: 100 개
본문 결측 수: 0 개
공백뿐인 본문 수: 0 개
최종 문서 수: 100 개
공백 제거로 달라진 문서 수; 30 개
      char_count  token_count
min       250.00        57.00
mean      347.27        83.34
max       550.00       137.00
'안녕하세요 이웃님들 반갑습니다. 요즘 날씨가 많이 쌀쌀해졌죠? 요즘 계절에 입으면 딱 좋을 인생 경량 패딩 하나 소개해 드릴게요.  다 함께 go go go~~  제가 입어보고 좋아서 엄마께도 구매해드렸네요. 딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서 어디서 샀냐고 많이들 물어보는 옷입니다.  엄마도 제가 입은 것 보시더니 탐내셔서  주문해드렸어요. 영하로 내려가는 날씨에는 이것만 입기엔 얇지만 초겨울까지는 운동 갈때 안에 얇은 기능성 반팔 입고 요것만 입어도 꽤 따뜻해요. 색상도 디자인도 무난해서 사무실에 두고 입기도 좋고, 평소에 가볍게 나갈때 코디하기도 좋아요. 앞으로 코트를 입거나 할때 안에 내피로 활용하기도 딱일 듯요! 맘같아선 색깔별로 쟁이고 싶습니다.  20대와 50대 모두 아우르는 디자인이 무난하고 깔끔하면서 고급스러운 옷 추천합니다~~~  설명 끝! 좀 도움이 되셨나요? 그러면 좋아요 꾹 눌러 주시고 전 이만 총총총~~ 항상 여러 이웃님들께 감사드립니다. '
'안녕하세요 이웃님들 반갑습니다. 요즘 날씨가 많이 쌀쌀해졌죠? 요즘 계절에 입으면 딱 좋을 인생 경량 패딩 하나 소개해 드릴게요.  다 함께 go go go~~  제가 입어보고 좋아서 엄마께도 구매해드렸네요. 딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서 어디서 샀냐고 많이들 물어보는 옷입니다.  엄마도 제가 입은 것 보시더니 탐내셔서  주문해드렸어요. 영하로 내려가는 날씨에는 이것만 입기엔 얇지만 초겨울까지는 운동 갈때 안에 얇은 기능성 반팔 입고 요것만 입어도 꽤 따뜻

### 필수 1 답변 작성란


**Q1.** 리뷰를 DataFrame에 저장하면 리뷰 본문 자체도 숫자 데이터처럼 일정한 구조를 갖게 되나요?
- 아니다. ID, 본문 등 정해진 컬럼 구조가 있지만, 본문 안의 단어  수, 문장 길이, 표현 방식, 평가 대상은 리뷰마다 다르다.

**Q2.** `strip()`과 `split()`은 각각 정제와 토큰화 중 어떤 작업인가요? 공백 기준 토큰 수를 형태소 수라고 불러도 되나요?
- strip은()은 불필요한 앞뒤 공백을 정리하는 정제,
- split()은 문장을 단위별로 나누는 토큰화를 의미
- 공백 토큰 수는 형태소 수로 나눈 것은 아님
- 상품이는 공백기준으로 하나로 보이지만 형태소 관점에서는 상품 / 이로 구분할 수 있다.

**Q3.** 문장부호, 숫자, 부정 표현을 일괄 삭제하지 않은 이유는 무엇인가요?
- 문장 부호는 문장의 경계나 강조, 숫자는 사이즈나 가격, 부정 표현은 평가의 방향을 나타낼 수 있기 때문에 제거하지 않음
- 예를 들어 '좋지 않다'에서 않다를 제거하면 반대로 해석될 여지가 있다.

**Q4.** 본문이 있는 리뷰의 상품명이나 전체 라벨이 비어 있다는 이유만으로 그 리뷰를 반드시 제거해야 하나요?
- 아니다. 지금은 본문 정제와 토큰화가 목적이기 때문에 본문이 잇으면 사용할 수가 있다.
- 상품명별 가격 비교나 정답 라벨을 이용한 평가에서는 리뷰 본문이 없어도 별도로 데이터를 유지해야할 수도 있다.

---
## 필수 2. 형태소 분석의 역할과 문맥 보존 비교

### 문제 2-1. 같은 대상의 언급과 같은 감정은 같은 의미일까?

#### 문제 설명

교안의 짧은 예시와 첫 번째 실제 리뷰를 비교합니다. 명사만 남겼을 때 알 수 있는 정보와 사라지는 정보를 구분하고, 세 파일이 같은 리뷰를 어떻게 설명하는지 확인하세요.

다음은 **교안의 개념을 설명하기 위해 사람이 작성한 분해 예시**입니다. 형태소 분석기를 실행한 결과가 아니며, 문장 전체의 형태소·품사를 나열한 것도 아닙니다.

| 표현 | 명사 | 조사 |
|---|---|---|
| 상품이 | 상품 | 이 |
| 상품은 | 상품 | 은 |
| 배송이 | 배송 | 이 |
| 배송은 | 배송 | 은 |

#### 요구사항

1. 문장 A `상품이 좋아요. 배송이 빨라요.`와 문장 B `상품은 좋지만 배송은 느려요.`를 공백 기준으로 나누어 출력하세요.
2. 위의 제공 예시에 따라 두 문장에 공통으로 등장하는 명사를 `common_nouns = ["상품", "배송"]`로 작성하여 출력하세요. 이 목록이 자동 추출 결과가 아님을 주석으로 명시하세요.
3. `documents`의 첫 번째 리뷰를 선택해 `review_id`와 `clean_text`를 출력하세요.
4. 아래 제공 함수를 사용하여 그 ID의 전체 라벨과 속성별 주석을 조회하세요. 전체 라벨 표의 `sentiment_code`, 속성별 표의 `aspect_no`, `aspect`, `evidence_text`, `sentiment_code`를 출력하세요.
5. 같은 리뷰에 연결된 전체 라벨 행 수와 속성별 주석 행 수를 출력하세요.

#### 해석 질문

**Q1.** 공백 토큰의 `상품이`와 `상품은`이 서로 다른 이유는 무엇이며, 형태소 분석은 어떤 도움을 줄 수 있나요?

**Q2.** 두 예시 문장에 같은 명사가 등장하면 감정도 같다고 볼 수 있나요? 무엇이 달라졌는지 설명하세요.

**Q3.** 첫 번째 실제 리뷰에서 만족 표현과 아쉬움 표현을 하나씩 골라 **속성명과 근거 문장**을 적으세요. 전체 라벨 하나로 모든 속성의 평가를 설명할 수 있나요?

**Q4.** 속성별 주석이 여러 행이면 여러 리뷰인가요? `aspect`를 원문에서 형태소 분석기로 자동 추출한 명사 목록이라고 해석해도 되나요?

#### 제출 결과

- 예시 문장별 공백 토큰과 제공 예시의 공통 명사
- 실제 리뷰 본문, 전체 라벨 표, 속성별 근거 표, 각 표의 행 수
- Q1~Q4 답변


### 필수 2 출력 읽기 도움말

짧은 문장 두 개에서는 같은 대상이 등장하는지와 같은 평가를 하는지를 나누어 봅니다. 공백 토큰에 조사·어미·문장부호가 붙어 있는지도 확인하세요.

실제 리뷰는 ID로 선택한 뒤 **속성 → 근거 문장 → 원문 앞뒤 조건**을 확인합니다. 전체 라벨 행 수와 속성 주석 행 수가 달라도 서로 다른 리뷰를 조회했다고 바로 판단하지 않습니다. 공통 명사 목록은 문제의 예시를 읽고 작성한 것이며 자동 추출 결과라고 설명하지 않습니다.

### 라벨 조회용 제공 코드

아래 함수는 같은 `review_id`의 행을 각 파일에서 찾아주는 보조 코드입니다. **함수 작성 자체는 평가하지 않습니다.** 그대로 실행하고 필수 2와 과제 1에서 재사용하세요.

`get_label_rows(리뷰_ID)`는 `(전체 라벨 표, 속성별 주석 표)`를 돌려줍니다. 두 표는 한 행의 의미가 다르므로 별도로 유지합니다. 기존 라벨을 조회하는 함수이며, 감정을 예측하는 함수가 아닙니다.


In [15]:
def get_label_rows(review_id):
    whole_rows = sample_review_labels[
        sample_review_labels["review_id"] == review_id
    ].copy()
    aspect_rows = sample_aspect_labels[
        sample_aspect_labels["review_id"] == review_id
    ].sort_values("aspect_no").copy()
    return whole_rows, aspect_rows


In [32]:
# 필수 2 코드를 작성하세요.
a = "상품이 좋아요. 배송이 빨라요."
b = "상품은 좋지만 배송은 느려요."

print('a 공백 기준 나누어 출력',a.split())
print('b 공백 기준 나누어 출력',b.split())

# 두 문장에 공통으로 등장하는 명사, 자동 추출 결과 아님
common_nouns = ['상품','배송']
print('두 문장 공통으로 등장하는 명사:',common_nouns)

print('documents 첫 번째 review의 review_id:',documents.loc[0]['review_id'])
print('documents 첫 번째 review의 clean_text:',documents.loc[0]['clean_text'])

sentiment_code, aspect_rows = get_label_rows(documents.loc[0]['review_id'])

print('전체 라벨: ',sentiment_code)
print('\n')
print('[속성별 표]\n',aspect_rows[['aspect_no','aspect','evidence_text','sentiment_code']])


a 공백 기준 나누어 출력 ['상품이', '좋아요.', '배송이', '빨라요.']
b 공백 기준 나누어 출력 ['상품은', '좋지만', '배송은', '느려요.']
두 문장 공통으로 등장하는 명사: ['상품', '배송']
documents 첫 번째 review의 review_id: 1024338
documents 첫 번째 review의 clean_text: 안녕하세요 이웃님들 반갑습니다. 요즘 날씨가 많이 쌀쌀해졌죠? 요즘 계절에 입으면 딱 좋을 인생 경량 패딩 하나 소개해 드릴게요.  다 함께 go go go~~  제가 입어보고 좋아서 엄마께도 구매해드렸네요. 딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서 어디서 샀냐고 많이들 물어보는 옷입니다.  엄마도 제가 입은 것 보시더니 탐내셔서  주문해드렸어요. 영하로 내려가는 날씨에는 이것만 입기엔 얇지만 초겨울까지는 운동 갈때 안에 얇은 기능성 반팔 입고 요것만 입어도 꽤 따뜻해요. 색상도 디자인도 무난해서 사무실에 두고 입기도 좋고, 평소에 가볍게 나갈때 코디하기도 좋아요. 앞으로 코트를 입거나 할때 안에 내피로 활용하기도 딱일 듯요! 맘같아선 색깔별로 쟁이고 싶습니다.  20대와 50대 모두 아우르는 디자인이 무난하고 깔끔하면서 고급스러운 옷 추천합니다~~~  설명 끝! 좀 도움이 되셨나요? 그러면 좋아요 꾹 눌러 주시고 전 이만 총총총~~ 항상 여러 이웃님들께 감사드립니다.
전체 라벨:     review_id  sentiment_code
0    1024338             1.0


[속성별 표]
    aspect_no aspect                                 evidence_text  \
0          1    디자인    딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서   
1          2     두께                                   이것만 입기엔 얇지만   
2  

### 필수 2 답변 작성란

**Q1.** 공백 토큰의 `상품이`와 `상품은`이 서로 다른 이유는 무엇이며, 형태소 분석은 어떤 도움을 줄 수 있나요?
- 문자열 끝에 붙은 조사가 '이'와 '은'으로 다르기 때문이다.
- 공백으로 나누면 조사도 토큰 안에 남기 때문에 형태소 분석으로 명사 상품과 조사를 구분하면 같은 대상을 일관된 기준으로 살펴보는 데 도움이 된다.

**Q2.** 두 예시 문장에 같은 명사가 등장하면 감정도 같다고 볼 수 있나요? 무엇이 달라졌는지 설명하세요.
- 아닙니다. 상품 A는 상품도 좋고, 배송도 빠르다는 내용이었고, B는 상품은 좋지만 배송이 느리다는 내용이ㅏㄷ.
- 두 문장을 [상품, 배송]으로 바꾸면 빨라요, 느려요의 차이 대조적인 관계가 사라진다.
- 같은 대상을 언급하여 보는 것과 같은 평가를 내리는 것은 다른 개념이다.

**Q3.** 첫 번째 실제 리뷰에서 만족 표현과 아쉬움 표현을 하나씩 골라 **속성명과 근거 문장**을 적으세요. 전체 라벨 하나로 모든 속성의 평가를 설명할 수 있나요?
- 만족한 예시에서는 디자인의 '기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서'입니다.
- 아쉬움 예시에서는 두께 부분에서 '이것만 입기에는 얇지만'이라는 문장이 등장한다.
- 이 리뷰는 전체 코트가 하나로 정리 되어 있지만 대상과 조건에 따라 만족과 아쉬움이 공존한다.
- 전체 라벨 하나를 모든 속성에 그대로 복사해서 해석할 수는 없다.

**Q4.** 속성별 주석이 여러 행이면 여러 리뷰인가요? `aspect`를 원문에서 형태소 분석기로 자동 추출한 명사 목록이라고 해석해도 되나요?
- 속성별 주석이 여러 행이라고 해서 여러 리뷰이지는 않다.
- 할 수 없다.

---
## 과제 1. 목적에 맞는 입력 준비와 문서 구조화

### 문제 3-1. 다른 리뷰에서도 본문과 관련 정보를 구분할 수 있을까?

#### 문제 설명

이번에는 `documents`의 **두 번째 리뷰(`iloc[1]`)**로 같은 절차를 독립적으로 수행하세요. 난이도는 필수 문제와 같으며, 새 모델이나 외부 API를 사용하지 않습니다.

#### 요구사항

1. 두 번째 리뷰의 ID와 정제된 본문을 출력하세요.
2. `get_label_rows()`로 해당 리뷰의 전체 라벨과 속성별 주석을 조회해 출력하세요.
3. 한 행짜리 `task_document`를 만드세요. 컬럼 순서는 `review_id`, `source`, `domain`, `category`, `product_name`, `review_text`, `clean_text`, `space_tokens`, `char_count`, `token_count`, `review_sentiment_code`로 하세요. 마지막 컬럼에는 **조회한 기존 전체 라벨 코드**를 넣으세요. 기존 코드가 결측이면 그대로 유지합니다.
4. 원문을 따로 보존한 상태에서 `task_document`에 `llm_input` 컬럼을 추가하고, 정제된 **문장 전체**를 넣으세요. 원문 의미를 보존한 입력을 준비하는 것이며 API는 호출하지 않습니다.
5. 별도 표 `task_aspects`에는 `review_id`, `aspect_no`, `aspect`, `evidence_text`, `sentiment_code`를 남기세요. 속성별 행을 문서 표에 펼쳐 붙이지 마세요.
6. 두 표의 컬럼명과 행 수, 문서의 `space_tokens` 앞 10개, `llm_input` 전문을 출력하세요. 본문에서 긍정적인 표현과 부정적인 표현을 하나씩 찾아 아래 Q2에 적으세요.

> 힌트: `documents.iloc[[1]]`은 한 행짜리 표를 선택합니다. `documents.iloc[1]`은 한 행의 값을 선택합니다. 기존 전체 라벨은 조회된 표의 `sentiment_code`에서 가져오세요. 제공 데이터에서는 리뷰당 전체 라벨 행이 하나입니다.

#### 해석 질문

**Q1.** `task_document`의 본문, 메타데이터, 기존 라벨을 구분하세요. 기존 라벨을 이번 실습에서 생성한 분석 결과라고 불러도 되나요?

**Q2.** 이 리뷰의 긍정 표현과 부정 표현을 각각 속성명·근거 문장과 함께 적으세요. 불만 이유를 요약할 입력으로 공백 토큰 목록과 정제된 문장 전체 중 무엇을 선택하겠나요? 그 이유는 무엇인가요?

**Q3.** ‘단어별 중요도 계산’, ‘표현은 달라도 의미가 비슷한 리뷰 검색’, ‘지시사항에 따른 불만 유형 분류·요약’에 적합한 방법과 주요 출력 형태를 각각 적으세요. 구현할 필요는 없습니다.

**Q4.** 강의의 전체 분석 과정 네 단계를 순서대로 적고, 이번 과제에서 직접 구현한 부분과 아직 구현하지 않은 부분을 구분하세요.

#### 제출 결과

- 요구사항 1~6의 코드와 출력
- 한 행짜리 `task_document`와 별도 `task_aspects`
- Q1~Q4 답변

CSV를 새로 저장할 필요는 없습니다. 결과 표와 답변을 노트북에 남기세요.


### 과제 시작 전 확인

앞 실습에서 만든 documents와 조회 함수를 그대로 활용합니다. 바뀌는 것은 대상 리뷰입니다. `.iloc[1]`은 두 번째 행의 값, `.iloc[[1]]`은 두 번째 행을 담은 한 행짜리 표라는 차이를 먼저 확인하세요.

원문·메타데이터·전처리 결과·기존 라벨을 구분해서 보관하고, llm_input에는 이후 분석에 사용할 본문을 넣습니다. 문서 표와 속성 표의 행 수를 각각 세고, 같은 리뷰를 가리키는 ID가 유지되는지 확인하세요. 새 모델을 구현하는 과제가 아닙니다.

In [47]:
# 과제 1 코드를 작성하세요.
second = documents.iloc[1]

print('두 번째 리뷰 ID:',second['review_id'])
print('두 번째 정제된 본문:',second['clean_text'])

whole_rows_two, aspect_rows_two = get_label_rows(second['review_id'])

print('전체 라벨:',whole_rows_two)
print('\n')
print('[속성별 표]\n',aspect_rows_two[['aspect_no','aspect','evidence_text','sentiment_code']])

doc_cols = ['review_id','source','domain','category','product_name','review_text','clean_text','space_tokens','char_count','token_count']
task_document = documents.iloc[[1]][doc_cols].reset_index(drop=True)
task_document['review_sentiment_code']=whole_rows_two['sentiment_code'].iloc[0]

task_document['llm_input']=task_document['clean_text']

task_aspects = aspect_rows_two[
    ["review_id", "aspect_no", "aspect", "evidence_text", "sentiment_code"]
].reset_index(drop=True)

print('task_document의 컬럼명:',task_document.columns)
print('task_document의 행:',len(task_document))
print('task_aspects의 컬럼명:',task_aspects.columns)
print('task_aspects의 행:',len(task_aspects))

print('앞 10개 토큰:',task_document.loc[0,'space_tokens'][:10])
print('llm_input 전문:',task_document.loc[0,'llm_input'])




두 번째 리뷰 ID: 1024477
두 번째 정제된 본문: 드디어 겨울이 찾아왔네요. 이제부터 슬슬 겨울 패딩 장만하셔야지요? 패딩 소개해 드릴게요.   작년부터 눈여겨 보던 OO입니다. 저는 블랙90 사이즈 구매했는데 딱 잘 맞네요. 작년에 비해 저렴하게 구매해서 아주 만족해요. 신제품은 경량패딩으로 되서 3in1기능이더라구요. 경량 패딩 구매 따로 하셔야 하는 분들에겐 신제품 매장에서 확인하시라고 권하고 싶네요. 저는 경량이 있는 관계로 이 제품 구매했는데 슬림핏이라 날씬해 보여요. 하지만 안에 두껍게 입기엔 좀 불편하더라구요. 간편하게 니트 티셔츠에 입기에는 딱입니다. 아직은 초겨울이라 OO를 쭉 입고 있어요. 후기 읽으며 많이 고민했고 도움을 받아서 저도 후기 남겨봅니다.누군가에겐 도움이 되셨기를 바라는 마음에서요.   OO 패딩 좋은 건 다들 아실테고.. 롱이면서 슬림하니 감추고 싶은 부분 감추고 그러면서도 날씬하게 보이는 패딩으로 이번 겨울 멋쟁이가 되어 보시길요..
전체 라벨:    review_id  sentiment_code
1    1024477             1.0


[속성별 표]
     aspect_no aspect                                      evidence_text  \
10          1    사이즈                         저는 블랙90 사이즈 구매했는데 딱 잘 맞네요.   
11          2     가격                           작년에 비해 저렴하게 구매해서 아주 만족해요   
12          3      핏                                    슬림핏이라 날씬해 보여요.    
13          4    활용성                              안에 두껍게 입기엔 좀 불편하더라구요.   
14          5    활용성                            간편하

### 과제 1 답변 작성란

**Q1.** `task_document`의 본문, 메타데이터, 기존 라벨을 구분하세요. 기존 라벨을 이번 실습에서 생성한 분석 결과라고 불러도 되나요?
-  task_document의 본문: review_text 전문,
- 메타데이터: review_id, source, domain, 
- 기존 라벨: 
- 기존 라벨을 이번 실습에서 생성한 분석 결과라고 할 수 없다.


**Q2.** 이 리뷰의 긍정 표현과 부정 표현을 각각 속성명·근거 문장과 함께 적으세요. 불만 이유를 요약할 입력으로 공백 토큰 목록과 정제된 문장 전체 중 무엇을 선택하겠나요? 그 이유는 무엇인가요?
- 이 리뷰에서 긍정 표현의 속성명은 사이즈이고 근거 문장은 '저는 블랙90 사이즈 구매했는데 딱 잘 맞네요.' 입니다.

**Q3.** ‘단어별 중요도 계산’, ‘표현은 달라도 의미가 비슷한 리뷰 검색’, ‘지시사항에 따른 불만 유형 분류·요약’에 적합한 방법과 주요 출력 형태를 각각 적으세요. 구현할 필요는 없습니다.
- 

**Q4.** 강의의 전체 분석 과정 네 단계를 순서대로 적고, 이번 과제에서 직접 구현한 부분과 아직 구현하지 않은 부분을 구분하세요.
- 

## 개념 정리 — 이번 실습을 마친 뒤 구분할 것

| 구분 | 이번 실습에서 확인한 핵심 |
|---|---|
| 정제와 토큰화 | strip은 양 끝을 정리하고, split은 문자열을 조각의 목록으로 나눕니다. |
| 공백 토큰과 형태소 | 상품이는 공백 토큰 하나이지만 상품과 이로 구분해 설명할 수 있습니다. |
| 원문과 분석용 입력 | review_text를 남겨 두고 clean_text·llm_input을 별도로 관리합니다. |
| 전체 라벨과 속성 라벨 | 리뷰 전체의 코드와 특정 대상·근거에 붙은 코드는 단위가 다릅니다. |
| 문서 수와 주석 수 | 리뷰 한 개에 주석 여러 개가 있을 수 있습니다. |
| 숫자 코드와 측정값 | sentiment_code는 범주 이름표, char_count는 길이를 센 값입니다. |
| 코드 실행과 의미 검증 | 오류 없이 실행되어도 원문 의미를 잘 보존했는지는 따로 읽어 확인합니다. |
| 입력 준비와 모델 실행 | 입력 표를 만들었어도 TF-IDF·임베딩·LLM 분석을 실행한 것은 아닙니다. |

### 분석 목적과 준비할 입력 연결하기

단어별 특징을 살피려면 어떤 단위로 나눌지 정하는 것이 필요합니다. 비슷한 의미의 글을 찾는 목적은 텍스트를 수치 벡터로 표현하는 임베딩과 연결되고, 지시에 따라 분류·요약하는 목적은 문맥을 보존한 본문을 LLM에 전달하는 방식과 연결됩니다. **이번 강의에서는 이 역할 구분까지만** 확인하며 추가 모델 구현은 요구하지 않습니다.

전처리 선택의 기준은 ‘얼마나 많이 지웠는가’가 아니라 ‘해결할 문제에 필요한 정보를 남겼는가’입니다. 가격 비교의 숫자, 부정 표현, 착용 조건은 이후 해석에 필요할 수 있습니다. 제거하기 전에 그 표현이 담당하는 역할부터 확인하세요.

---
## 실습 마무리

1. 이번 실습에서 해결하려고 한 텍스트 처리 문제는 무엇이었나요?
2. 정제·토큰화·형태소 분석은 어떻게 다른가요?
3. 전처리 후 필요한 정보가 보존되었는지는 무엇을 근거로 확인했나요?
4. 키워드 분석용 입력과 LLM 분류·요약용 입력을 반드시 같게 만들어야 하나요?
5. 리뷰 한 개와 속성별 주석 여러 개를 어떻게 연결하고 구분했나요?

각 질문에 1~3문장으로 답하세요.


### 실습 마무리 답변 작성란

1.

2.

3.

4.

5.
